# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the [FAIR²](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library, leveraging the Croissant schema to ensure reproducible, interpretable data ingest and processing.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

print(f"Dataset Name: {dataset.metadata.name}\n")
print(f"Description:\n{dataset.metadata.description}\n")
print(f"FAIR Identifier: {metadata.get('identifier', 'N/A')}")
print(f"Authors: {metadata.get('author', 'N/A')}")
print(f"Version: {metadata.get('version', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, columns, and their IDs.

We'll use the Croissant metadata to discover all `RecordSet` objects for inspection. All references will use their `@id` as required for programmatic integrity.

In [ ]:
from pprint import pprint

# List the available record sets by `@id`, name, and description
record_sets = list(dataset.record_sets)
print("Available Record Sets:")
for record_set in record_sets:
    print(f"- @id: {record_set.id} | name: {getattr(record_set, 'name', 'N/A')} | description: {getattr(record_set, 'description', '')}")

In [ ]:
# For each record set, list its fields and columns by @id
for rs in record_sets:
    print(f"\nRecord Set: {rs.id} | name: {getattr(rs, 'name', 'N/A')}")
    print("Fields:")
    for field in getattr(rs, 'fields', []):
        print(f"    - @id: {getattr(field, 'id', '')} | name: {getattr(field, 'name', '')} | type: {getattr(field, 'data_type', '')}")
    print("Columns:")
    for col in getattr(rs, 'columns', []):
        print(f"    - @id: {getattr(col, 'id', '')} | name: {getattr(col, 'name', '')} | dtype: {getattr(col, 'data_type', '')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

**Note:** The exact `@id`s must be taken from the metadata above and used directly as variables.

In [ ]:
# Example: List all available record_set @ids
record_set_ids = [rs.id for rs in record_sets]

# Load data for all record sets found
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for RecordSet @id: {record_set_id}, shape: {df.shape}")
    else:
        print(f"No records found for RecordSet @id: {record_set_id}")

In [ ]:
# Display columns and first records for the main tabular record set
# Update the following with the main record set @id containing tabular protein data
main_record_set_id = record_set_ids[0] if len(record_set_ids) > 0 else None
if main_record_set_id and main_record_set_id in dataframes:
    print(f"First few columns in DataFrame for record set {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records, normalizing numeric fields, and categorizing data. Operations can include removing outliers, transforming data, or grouping by key attributes. All fields must be referenced by their `@id` as per Croissant guidelines.

In [ ]:
# Identify a numeric field by @id (use data overview to select a suitable column @id)
# We'll use a common mass spectrometry column: e.g. 'cr:field:Coverage' or similar, replace with actual
numeric_field_id = None
for col in dataframes[main_record_set_id].columns:
    # Guess likely numeric fields by name pattern, you may need to adjust this to the dataset
    if 'coverage' in col.lower() or 'abundance' in col.lower() or 'mw' in col.lower() or 'count' in col.lower():
        numeric_field_id = col
        break

if numeric_field_id is None:
    print("No numeric field detected. Please set 'numeric_field_id' manually to a numeric @id column.")
else:
    print(f"Using numeric field: {numeric_field_id}")

    # Filter on a threshold value (e.g. > 10)
    threshold = 10
    filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize
    normcol = f"{numeric_field_id}_normalized"
    filtered_df[normcol] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, normcol]].head())

    # Group by protein or similar field if available (such as 'cr:field:Accession' or 'cr:field:Description')
    # Try to find a string field for grouping
    group_field = None
    for col in dataframes[main_record_set_id].columns:
        if 'accession' in col.lower() or 'description' in col.lower() or 'id' in col.lower():
            group_field = col
            break

    if group_field:
        print(f"Grouping by field: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"Grouped means by {group_field}:")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib/seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if numeric_field_id is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(dataframes[main_record_set_id][numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()
    
    # If grouped_df exists, plot top 10 groups by mean value
    if 'grouped_df' in locals() and not grouped_df.empty:
        plt.figure(figsize=(10,5))
        top_n = grouped_df.sort_values(numeric_field_id, ascending=False).head(10)
        sns.barplot(x=numeric_field_id, y=group_field, data=top_n, palette='viridis')
        plt.title(f'Top 10 {group_field} by mean {numeric_field_id}')
        plt.xlabel(f"Mean {numeric_field_id}")
        plt.ylabel(group_field)
        plt.show()

## 6. Conclusion
This notebook provided an overview and basic analysis of the FAIR² dataset. We loaded metadata, explored available record sets and fields, extracted protein-related data, performed normalization and grouping by key protein identifiers, and visualized distributions.

Further analyses can include deeper grouping, advanced filtering, or more domain-specific analytics (e.g., by peptide sequences or modifications). The Croissant schema ensures that all field and record set references remain robust and reproducible via their `@id` values.